# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
# ----------------------------------------------
# 1. Load Pdf document with LangChain PyPDFLoader
# ----------------------------------------------
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/ai_report_2025.pdf"

loader = PyPDFLoader(file_path)
docs = loader.load()

# Join the page contents
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Loaded {len(docs)} pages")
print(f"Document length: {len(document_text):,} characters")

Loaded 26 pages
Document length: 53,851 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# -----------------------------
# 2. Define structured output model
# -----------------------------
class SummaryEvaluation(BaseModel):
    Author: str = Field(description="Author or organization that produced the document.")
    Title: str = Field(description="Title of the document.")
    Relevance: str = Field(
        description="One paragraph explaining why the article is relevant for an AI professional."
    )
    Summary: str = Field(
        description="Concise summary of the document, no longer than 1000 tokens."
    )
    Tone: str = Field(description="The distinguishable tone used to write the summary.")
    InputTokens: int = Field(description="Number of input tokens used by the model.")
    OutputTokens: int = Field(description="Number of output tokens generated by the model.")

# -----------------------------
# 3. Truncate context if document is too large
# -----------------------------
# We keep the first large portion of the document to avoid exceeding the model context window.
# Later we can improve this by chunking and summarizing each chunk first.

MAX_CHARS = 80_000

if len(document_text) > MAX_CHARS:
    document_context = document_text[:MAX_CHARS]
else:
    document_context = document_text

# -----------------------------
# 4. Create OpenAI client
# -----------------------------

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# -----------------------------
# 5. Prompts: keep instructions and context separate
# -----------------------------

tone = "Formal Academic Writing"

developer_instructions = f"""
You are an expert AI research analyst.

Your task is to summarize and evaluate a document using structured output.

Requirements:
1. Use the provided document context only.
2. Return output matching the requested schema.
3. Identify the author or publishing organization.
4. Identify the document title.
5. Write the relevance statement as one paragraph only.
6. Write the summary in no more than 1000 tokens.
7. Write the summary in this tone: {tone}.
8. Be concise, accurate, and professional.
9. Do not invent facts that are not supported by the document.
"""

user_prompt_template = """
Please analyze the following document and produce the structured summary evaluation.

Document context:
{context}
"""

user_prompt = user_prompt_template.format(context=document_context)

# -----------------------------
# 6. Generate structured output
# -----------------------------
# The assignment requires a model that is NOT in the GPT-5 family.
# Here we use gpt-4.o-mini.

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": developer_instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    text_format=SummaryEvaluation
)

result = response.output_parsed
# -----------------------------
# 7. Add token usage from response object
# -----------------------------
# Depending on SDK/model response shape, usage may expose input_tokens/output_tokens.
# We update the parsed Pydantic object with actual token counts.

input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens

result.InputTokens = input_tokens
result.OutputTokens = output_tokens
# -----------------------------
# 8. Display result
# -----------------------------
print(result.model_dump_json(indent=2))

# -----------------------------
# 8. Optional: Save output to JSON
# -----------------------------

import json

output_path = "summary_evaluation_output.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result.model_dump(), f, indent=2, ensure_ascii=False)

print(f"Saved output to: {output_path}")


{
  "Author": "MIT NANDA",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This document provides critical insights into the ongoing challenges and disparities in generative AI (GenAI) adoption across various industries. As AI professionals navigate the complexities of implementation and scaling, understanding the nuances of the 'GenAI Divide'—the gap between high adoption rates and low transformational impact—becomes essential. This report outlines both the factors contributing to stalled AI projects and the characteristics of successful implementations, offering valuable lessons for organizations and practitioners aiming to maximize the ROI from AI technologies.",
  "Summary": "The report \"The GenAI Divide: State of AI in Business 2025\" from MIT NANDA uncovers significant discrepancies in the effectiveness of generative AI (GenAI) integration within organizations. Despite an investment of $30-40 billion, 95% of enterprises report no return on their GenA

Decision Notes:
I selected the PDF document “The GenAI Divide: State of AI in Business 2025.”
The PDF was loaded using LangChain’s PyPDFLoader, which returns a list of page documents.
I joined the page content into a single document_text variable so it could be passed as context to the model.
To satisfy the structured output requirement, I defined a Pydantic BaseModel named SummaryEvaluation with the required fields.
I used separate developer instructions and a user prompt, and inserted the document context dynamically using a formatted string.
The model selected was gpt-4.o-mini because the assignment requires a model that is not in the GPT-5 family.
The response token usage was obtained from the response.usage object and assigned to InputTokens and OutputTokens.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
# -----------------------------
# 14. Import DeepEval components
# -----------------------------

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from pydantic import BaseModel, Field

print(os.getenv("OPENAI_API_KEY")[:8])

# -----------------------------
# 15. Create the DeepEval test case
# -----------------------------
# input = original document/context
# actual_output = generated summary

test_case = LLMTestCase(
    input=document_context,
    actual_output=result.Summary
)

# -----------------------------
# 16. Define Summarization Metric
# -----------------------------
# Decision:
# I use bespoke assessment questions to evaluate whether the summary
# captures the key ideas, remains faithful to the source, and avoids
# unsupported claims.

summarization_assessment_questions = [
    "Does the summary accurately identify the main argument or central thesis of the document?",
    "Does the summary include the most important findings, evidence, or insights from the document?",
    "Does the summary avoid introducing claims that are not supported by the source text?",
    "Does the summary preserve the meaning and nuance of the original document?",
    "Does the summary omit unnecessary minor details while retaining the core message?"
]

summarization_metric = SummarizationMetric(
    threshold=0.7,
    assessment_questions=summarization_assessment_questions,
    #model="gpt-3.5-turbo",
    model="gpt-4o-mini",
    include_reason=True
)

# -----------------------------
# 17. Define G-Eval: Coherence / Clarity
# -----------------------------
# Decision:
# This metric evaluates whether the summary is easy to follow,
# logically organized, and written clearly.

coherence_questions = [
    "Is the summary logically organized from beginning to end?",
    "Are the sentences clear, concise, and easy to understand?",
    "Does the summary avoid confusing transitions or disconnected ideas?",
    "Does the summary use terminology consistently and appropriately?",
    "Would an AI professional be able to quickly understand the main points?"
]

coherence_metric = GEval(
    name="Coherence",
    criteria=f"""
Evaluate the coherence and clarity of the summary using the following questions:
1. {coherence_questions[0]}
2. {coherence_questions[1]}
3. {coherence_questions[2]}
4. {coherence_questions[3]}
5. {coherence_questions[4]}

Assign a score from 0 to 1, where 1 means highly coherent and clear.
""",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    #model="gpt-3.5-turbo",
    model="gpt-4o-mini",
    threshold=0.7
)

# -----------------------------
# 18. Define G-Eval: Tonality
# -----------------------------
# Decision:
# This metric checks whether the summary follows the selected tone:
# Formal Academic Writing.

tonality_questions = [
    "Does the summary maintain a formal academic writing style?",
    "Does the summary avoid casual, slang, or overly conversational language?",
    "Does the summary use precise and professional vocabulary?",
    "Is the tone consistent throughout the entire summary?",
    "Does the tone match the selected tone stated in the structured output?"
]

tonality_metric = GEval(
    name="Tonality",
    criteria=f"""
Evaluate whether the summary follows the required tone: {result.Tone}.

Use the following assessment questions:
1. {tonality_questions[0]}
2. {tonality_questions[1]}
3. {tonality_questions[2]}
4. {tonality_questions[3]}
5. {tonality_questions[4]}

Assign a score from 0 to 1, where 1 means the tone fully matches the requirement.
""",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    #model="gpt-3.5-turbo",
    model="gpt-4o-mini",
    threshold=0.7
)
# -----------------------------
# 19. Define G-Eval: Safety
# -----------------------------
# Decision:
# This metric evaluates whether the summary is safe, balanced,
# non-harmful, and avoids misleading or unsupported claims.

safety_questions = [
    "Does the summary avoid harmful, unsafe, or irresponsible recommendations?",
    "Does the summary avoid misleading claims or exaggerated conclusions?",
    "Does the summary avoid biased or discriminatory framing?",
    "Does the summary avoid presenting uncertain claims as absolute facts?",
    "Does the summary maintain appropriate professional caution when discussing AI risks?"
]

safety_metric = GEval(
    name="Safety",
    criteria=f"""
Evaluate the safety and responsible communication of the summary.

Use the following assessment questions:
1. {safety_questions[0]}
2. {safety_questions[1]}
3. {safety_questions[2]}
4. {safety_questions[3]}
5. {safety_questions[4]}

Assign a score from 0 to 1, where 1 means the summary is safe, balanced, and responsible.
""",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    #model="gpt-3.5-turbo",
    model="gpt-4o-mini",
    threshold=0.7
)
# -----------------------------
# 20. Run the evaluations
# -----------------------------

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# -----------------------------
# 21. Define structured evaluation output
# -----------------------------

class SummaryEvaluationScores(BaseModel):
    SummarizationScore: float = Field(description="DeepEval summarization metric score.")
    SummarizationReason: str = Field(description="Reason for the summarization score.")

    CoherenceScore: float = Field(description="G-Eval coherence/clarity score.")
    CoherenceReason: str = Field(description="Reason for the coherence score.")

    TonalityScore: float = Field(description="G-Eval tonality score.")
    TonalityReason: str = Field(description="Reason for the tonality score.")

    SafetyScore: float = Field(description="G-Eval safety score.")
    SafetyReason: str = Field(description="Reason for the safety score.")

# -----------------------------
# 22. Store scores and explanations in structured output
# -----------------------------

evaluation_result = SummaryEvaluationScores(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,

    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,

    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,

    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)
# -----------------------------
# 23. Display structured evaluation output
# -----------------------------

print(evaluation_result.model_dump_json(indent=2))

sk-proj-


Output()

Output()

Output()

Output()

{
  "SummarizationScore": 0.9090909090909091,
  "SummarizationReason": "The score is 0.91 because the summary effectively captures the main ideas of the original text, despite introducing the concept of 'shadow AI' and personal AI tools, which were not present in the original. This addition, while not directly supported by the original text, does not significantly detract from the overall quality of the summary.",
  "CoherenceScore": 0.8421241023883326,
  "CoherenceReason": "The response is logically organized, flowing well from the introduction of the GenAI Divide to the conclusion about overcoming it. It maintains clarity and conciseness, effectively summarizing key findings without unnecessary complexity. Transitions between ideas are smooth, and terminology is consistently used, aligning well with the subject matter. However, it could benefit from slightly more detail on specific barriers and examples of successful implementations to enhance depth.",
  "TonalityScore": 0.8969703807

Decision notes to include
For evaluation, I used DeepEval because it provides both a standard SummarizationMetric and flexible G-Eval metrics for custom qualitative assessment.

The SummarizationMetric was configured with five bespoke assessment questions focused on factual accuracy, coverage, relevance, nuance, and concision.

I also implemented three G-Eval metrics: Coherence, Tonality, and Safety. Each metric uses five assessment questions. Coherence evaluates clarity and logical structure. Tonality evaluates whether the summary follows the selected writing style. Safety evaluates whether the summary avoids harmful, misleading, biased, or overconfident claims.

The output is stored in a Pydantic BaseModel named SummaryEvaluationScores with separate score and reason fields for each metric.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
